# **Fine tuning de asistente legal usando Lora**

**Tópicos Especiales y Aplicaciones en IA** — Universidad EAFIT — Módulo 1 — Transformers

Integrantes:
- Martin Valencia
- Pablo Cabrejos
- Samuel Lopez
- Miguel Ortiz
---


## 0. Setup


In [1]:
# Instalamos lo necesario para LoRA, datasets y métricas.
%pip install -q -U peft datasets evaluate accelerate scikit-learn
%pip uninstall -y torchao
print('\nListo.')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

Listo.


- Se importan dependencias necesarias, se fija una semilla y se usa la GPU de collab:

In [2]:
import random
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
import transformers
import peft

SEED = 42

def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)
print("device:", device)

if device == "cpu":
    print(" Activa la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.")

c:\Users\USUARIO\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.15.0
peft: 0.20.0
torch: 2.11.0+cu126
device: cuda


## 1. Cargar, unir y limpiar datos

Vamos a unir **dasatet_cross_encoder.csv** con **diccionario_articulos.csv** para así lograr que las consultas queden relacionadas con las descripciones de los articulos

In [3]:
# Traemos los datos directamente del repositorio:

REPO = "Bosnape/cabrejos-ortiz-valencia-lopez"
REF = "main"  # Al entregar, conviene reemplazarlo por el commit final del equipo.
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data"

pares = pd.read_csv(f"{BASE_URL}/dataset_cross_encoder.csv")
diccionario = pd.read_csv(f"{BASE_URL}/diccionario_articulos.csv")

print("Pares originales:", len(pares))
display(pares.head(3))
display(diccionario.head(3))

Pares originales: 561


,consulta,articulo,tipo,label,sentencia_origen
0,Trabajé como operador de bus articulado desde ...,CST Art. 127,positivo,1,SL-3630/22
1,Trabajé como operador de bus articulado desde ...,CST Art. 128,positivo,1,SL-3630/22
2,Trabajé como operador de bus articulado desde ...,CST Art. 236,negativo_facil,0,SL-3630/22


,fuente,numero,texto_completo,n_citas_en_dataset,url_fuente
0,CGP,167,Artículo 167. Carga de la prueba. Incumbe a la...,1,https://www.funcionpublica.gov.co/eva/gestorno...
1,CGP,244,Artículo 244. Documento auténtico. Es auténtic...,1,https://www.funcionpublica.gov.co/eva/gestorno...
2,CPTSS,50,ARTICULO 50. -Extra y ultra petita. El juez (d...,1,https://www.funcionpublica.gov.co/eva/gestorno...


Ahora separamos la cita, por ejemplo CST Art. 127, en la llave (CST, 127) y hacemos el join con el texto completo del artículo.

In [4]:
def normalizar_llave(valor):
    """Normaliza tildes, mayúsculas y espacios para hacer joins robustos."""
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto).strip().upper()
    return texto

# Variantes de escritura de la misma fuente vistas en las citas que extrajo el LLM
# (ninguna sigue un formato único: "CST", "Código Sustantivo del Trabajo", "C S T", ...).
ALIASES_FUENTE = {
    "CODIGO SUSTANTIVO DEL TRABAJO": "CST",
    "CODIGO SUSTANTIVO DE TRABAJO": "CST",
    "C S T": "CST",
    "CONSTITUCION POLITICA": "CP",
}

def normalizar_fuente(valor):
    fuente = normalizar_llave(valor)

    if fuente.startswith("CODIGO SUSTANTIVO DEL TRABAJO"):
        return "CST"

    if fuente.startswith("CODIGO SUSTANTIVO DE TRABAJO"):
        return "CST"

    if fuente.startswith("CONSTITUCION POLITICA"):
        return "CP"

    if fuente == "C S T":
        return "CST"

    return ALIASES_FUENTE.get(fuente, fuente)

# Misma lógica que normalizar_articulos_citados() en ds_diccionario_articulos.ipynb,
# aquí aplicada fila a fila para poder hacer el join (fuente, numero) con el diccionario.
def parsear_articulo(articulo):
    texto = str(articulo).strip()

    # Ej.: CST Art. 127 / Ley 6 de 1945, Art. 1
    match = re.match(
        r"^(?P<fuente>.+?)(?:,)?\s+"
        r"Art(?:[íi]culo|\.)?\s*"
        r"(?P<numero>[0-9]+[A-Za-z]?)",
        texto,
        flags=re.IGNORECASE,
    )
    if match:
        return (
            normalizar_fuente(match.group("fuente")),
            normalizar_llave(match.group("numero")),
        )

    # Ej.: Artículo 57 numeral 5 del Código Sustantivo del Trabajo
    match = re.match(
        r"^Art(?:[íi]culo|\.)?\s*"
        r"(?P<numero>[0-9]+[A-Za-z]?)"
        r"(?:\s*,?\s*(?:numeral|num\.?)\s*[0-9]+[A-Za-z]?)?"
        r"\s+(?:del|de la|de el)\s+"
        r"(?P<fuente>.+?)\s*$",
        texto,
        flags=re.IGNORECASE,
    )
    if match:
        return (
            normalizar_fuente(match.group("fuente")),
            normalizar_llave(match.group("numero")),
        )

    # Norma sin artículo específico: no es error.
    return None, None


pares[["fuente_key", "numero_key"]] = pd.DataFrame(
    pares["articulo"].map(parsear_articulo).tolist(),
    index=pares.index,
)

diccionario["fuente_key"] = diccionario["fuente"].map(normalizar_fuente)
diccionario["numero_key"] = diccionario["numero"].map(normalizar_llave)

menciona_articulo = pares["articulo"].astype(str).str.contains(
    r"\bArt(?:[íi]culo|\.)?",
    case=False,
    regex=True,
)

errores_reales = pares[pares["fuente_key"].isna() & menciona_articulo]

normas_sin_articulo = pares[pares["fuente_key"].isna() & ~menciona_articulo]

print("Normas completas sin artículo específico:", len(normas_sin_articulo))
display(normas_sin_articulo[["articulo", "sentencia_origen"]].drop_duplicates())
if len(errores_reales) > 0:
    print(
        " Hay Citas con artículo que no se pudieron interpretar. "
    )
    display(errores_reales[["articulo", "sentencia_origen"]].drop_duplicates())

df = pares.merge(
    diccionario[["fuente_key", "numero_key", "fuente", "texto_completo"]],
    on=["fuente_key", "numero_key"],
    how="left",
)

sin_texto = df["texto_completo"].fillna("").astype(str).str.strip().eq("")
print("Pares sin texto normativo completo:", sin_texto.sum())
print("\nDistribución antes de retirar faltantes:")
display(df["label"].value_counts().sort_index())

# Un cross-encoder necesita el texto del artículo; una cita corta no es suficiente.
df = df.loc[~sin_texto].copy().reset_index(drop=True)

# Input final del segundo segmento del cross-encoder: texto_completo ya trae el número
# de artículo en su propio encabezado, pero no dice de qué norma es — por eso anteponemos
# la fuente (no el identificador completo, que sería redundante).
df["texto_input"] = df["fuente"].astype(str) + ". " + df["texto_completo"].astype(str)
df["label"] = df["label"].astype(int)

columnas_modelo = [
    "consulta", "texto_input", "label",
    "sentencia_origen", "articulo", "tipo"
]
df = df[columnas_modelo]

print("\nPares utilizables:", len(df))
print("Sentencias únicas:", df["sentencia_origen"].nunique())
display(df["label"].value_counts().sort_index())
display(df.head(2))


Normas completas sin artículo específico: 21


,articulo,sentencia_origen
27,Ley 1636 de 2013,SU-075/18
111,Ordenanza 008 de 1986,T-282/15
117,Decreto Ley 2351 de 1965,T-1040/06
157,Ley 790 de 2002,T-866/05
158,Ley 813 de 2003,T-866/05
181,Decreto 190 de 2003,T-206/06
208,Ley 1822 de 2017,T-043/20
221,Ley 361 de 1997,T-198/06
233,Ley 361 de 1997,T-195/22
256,Ley 361 de 1997,T-076/24


Pares sin texto normativo completo: 23

Distribución antes de retirar faltantes:


label
0    276
1    285
Name: count, dtype: int64


Pares utilizables: 538
Sentencias únicas: 138


label
0    276
1    262
Name: count, dtype: int64

,consulta,texto_input,label,sentencia_origen,articulo,tipo
0,Trabajé como operador de bus articulado desde ...,"CST. ARTICULO 127. Modificado por el art. 14, ...",1,SL-3630/22,CST Art. 127,positivo
1,Trabajé como operador de bus articulado desde ...,"CST. ARTICULO 128. Modificado por el art. 15, ...",1,SL-3630/22,CST Art. 128,positivo


Hay que recordar que ya se conocía el hecho de que había 21  artículos que no se podían relacionar ya que pese a que existía una sentencia, faltaba dicho articulo para poder vincularlo a una descripción, por lo que estas filas se descartan para el entrenamiento del modelo

# 1.1 separación del dataset: Split train / validation / test sin fuga

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# Primero separamos un test final: no se usa para escoger hiperparámetros.
split_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED,
)

# Agrupamos por sentencia_origen, no por fila: el positivo y sus negativos de una misma
# sentencia comparten la misma consulta — dividir por fila filtraría esa consulta a más
# de un split (fuga de información).
idx_train_val, idx_test = next(
    split_test.split(df, groups=df["sentencia_origen"])
)

train_val_df = df.iloc[idx_train_val].reset_index(drop=True)
test_df = df.iloc[idx_test].reset_index(drop=True)

# Del 85% restante, 15/85 equivale a 17.65%; deja ~70/15/15 total.
split_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=SEED,
)

idx_train, idx_val = next(
    split_val.split(train_val_df, groups=train_val_df["sentencia_origen"])
)

train_df = train_val_df.iloc[idx_train].reset_index(drop=True)
val_df = train_val_df.iloc[idx_val].reset_index(drop=True)

def resumen_split(nombre, datos):
    print(
        f"{nombre:10} | pares={len(datos):3} | "
        f"sentencias={datos['sentencia_origen'].nunique():3} | "
        f"positivos={datos['label'].sum():3} | "
        f"proporción positiva={datos['label'].mean():.3f}"
    )

resumen_split("train", train_df)
resumen_split("validation", val_df)
resumen_split("test", test_df)

# Verificación: ninguna sentencia debería estar en más de un conjunto.
grupos_train = set(train_df["sentencia_origen"])
grupos_val = set(val_df["sentencia_origen"])
grupos_test = set(test_df["sentencia_origen"])

sin_fuga = (
    grupos_train.isdisjoint(grupos_val)
    and grupos_train.isdisjoint(grupos_test)
    and grupos_val.isdisjoint(grupos_test)
)

print(f"\n{'✅' if sin_fuga else '⚠️'} Split agrupado, sin fuga entre sentencias: {sin_fuga}")

train      | pares=375 | sentencias= 96 | positivos=183 | proporción positiva=0.488
validation | pares= 87 | sentencias= 21 | positivos= 45 | proporción positiva=0.517
test       | pares= 76 | sentencias= 21 | positivos= 34 | proporción positiva=0.447

✅ Split agrupado, sin fuga entre sentencias: True


## 2. Modelo base + tokenizer, y tokenización

Cargamos el tokenizador de BETO y tokenizamos el dataset

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODELO = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODELO)

def tokenizar(batch):
    # truncation="only_second": BETO tiene un tope duro de 512 tokens. Algunos artículos
    # superan los 1800 tokens, pero la consulta nunca pasa de ~135 — recortamos solo el
    # artículo, nunca la consulta (ver data/README.md, sección de truncation).
    return tokenizer(
        batch["consulta"],
        batch["texto_input"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

# Conservamos los DataFrames originales: los necesitaremos para ranking y ejemplos.
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)  # held-out: no se toca hasta la sección 6

train_tok = train_ds.map(tokenizar, batched=True)
val_tok = val_ds.map(tokenizar, batched=True)
test_tok = test_ds.map(tokenizar, batched=True)

# Dejamos únicamente lo que el modelo necesita para entrenar.
columnas_modelo = {"input_ids", "attention_mask", "token_type_ids", "label"}

def quitar_metadatos(dataset):
    columnas_a_quitar = [
        col for col in dataset.column_names
        if col not in columnas_modelo
    ]
    return dataset.remove_columns(columnas_a_quitar)

train_tok = quitar_metadatos(train_tok)
val_tok = quitar_metadatos(val_tok)
test_tok = quitar_metadatos(test_tok)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

print(train_tok)
print(val_tok)
print(test_tok)

# .keys() solo muestra los campos del diccionario, no el contenido tokenizado en sí.
# Para ver las subpalabras reales hay que decodificar input_ids:
print("\nEjemplo tokenizado (primeras 20 subpalabras de input_ids):")
print(tokenizer.convert_ids_to_tokens(train_tok[0]["input_ids"])[:20])

Map: 100%|██████████| 76/76 [00:00<00:00, 2490.91 examples/s]


Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 375
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 87
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 76
})

Ejemplo tokenizado (primeras 20 subpalabras de input_ids):
['[CLS]', 'Trabaj', '##é', 'como', 'operador', 'de', 'bus', 'articul', '##ado', 'desde', 'septiembre', 'de', '2004', 'hasta', 'septiembre', 'de', '2014', '.', 'Me', 'paga']


## 3. Baselines (contra qué comparamos) (MODIFICADA)

La asignación pide dos referencias distintas: **clase mayoritaria** y **BETO sin afinar**.
La clase mayoritaria comprueba cuánto se logra sin usar el texto; BETO sin afinar usa el mismo
encoder del modelo final, pero con una cabeza binaria recién inicializada y sin entrenamiento en
esta tarea. La comparación principal se hará sobre el mismo conjunto de `validation` de Miguel.

In [7]:
# MÉTRICAS COMPARTIDAS (MODIFICADA)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metricas_desde_predicciones(predicciones, labels):
    """Calcula métricas binarias cuando ya se tienen las clases predichas."""
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predicciones,
        average="binary",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predicciones),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def metricas_clasificacion(logits, labels):
    """Convierte logits en clases y reutiliza el cálculo común de métricas."""
    predicciones = np.argmax(logits, axis=-1)
    return metricas_desde_predicciones(predicciones, labels)

def cargar_modelo_base():
    # Fija también la inicialización de la cabeza clasificadora.
    fijar_semilla(SEED)

    return AutoModelForSequenceClassification.from_pretrained(
        MODELO,
        num_labels=2,
    )

In [8]:
# BASELINE DE CLASE MAYORITARIA (CREADA)
# Baseline 1: predecir siempre la clase más frecuente observada en TRAIN.
# La clase se decide sin mirar validation ni test, para no usar información de evaluación.
clase_mayoritaria = int(train_df["label"].value_counts().idxmax())

pred_mayoritaria_val = np.full(len(val_df), clase_mayoritaria, dtype=int)
pred_mayoritaria_test = np.full(len(test_df), clase_mayoritaria, dtype=int)

metricas_mayoritaria_val = metricas_desde_predicciones(
    pred_mayoritaria_val,
    val_df["label"].to_numpy(),
)
metricas_mayoritaria_test = metricas_desde_predicciones(
    pred_mayoritaria_test,
    test_df["label"].to_numpy(),
)

nombre_clase = "RELEVANTE" if clase_mayoritaria == 1 else "NO RELEVANTE"
print("Distribución de clases en train:")
display(train_df["label"].value_counts().sort_index().rename("cantidad").to_frame())
print(f"Clase mayoritaria elegida solo con train: {clase_mayoritaria} ({nombre_clase})")
print("\n=== BASELINE: CLASE MAYORITARIA (validation) ===")
for nombre, valor in metricas_mayoritaria_val.items():
    print(f"{nombre:10}: {valor:.3f}")

Distribución de clases en train:


,cantidad
label,
0,192
1,183


Clase mayoritaria elegida solo con train: 0 (NO RELEVANTE)

=== BASELINE: CLASE MAYORITARIA (validation) ===
accuracy  : 0.483
precision : 0.000
recall    : 0.000
f1        : 0.000


In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Baseline exigido por M1 (STAI_Plantilla_Proyecto.md, sección 5): el mismo BETO sin
# fine-tuning, cabeza clasificadora recién inicializada — no la clase mayoritaria.
modelo_baseline = cargar_modelo_base().to(device)

args_baseline = TrainingArguments(
    output_dir="./baseline_temporal",
    per_device_eval_batch_size=16,
    report_to="none",
    seed=SEED,
)

trainer_baseline = Trainer(
    model=modelo_baseline,
    args=args_baseline,
    processing_class=tokenizer,
    data_collator=collator,
)

pred_baseline = trainer_baseline.predict(val_tok)
pred_baseline_test = trainer_baseline.predict(test_tok)

metricas_baseline_val = metricas_clasificacion(
    pred_baseline.predictions,
    pred_baseline.label_ids,
)
metricas_baseline_test = metricas_clasificacion(
    pred_baseline_test.predictions,
    pred_baseline_test.label_ids,
)

print("=== BASELINE: BETO SIN FINE-TUNING (validation) ===")
for nombre, valor in metricas_baseline_val.items():
    print(f"{nombre:10}: {valor:.3f}")

print("\n=== BASELINE: BETO SIN FINE-TUNING (test, held-out) ===")
for nombre, valor in metricas_baseline_test.items():
    print(f"{nombre:10}: {valor:.3f}")

# Liberamos GPU antes de cargar el modelo LoRA.
del modelo_baseline, trainer_baseline
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16407.75it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differe

=== BASELINE: BETO SIN FINE-TUNING (validation) ===
accuracy  : 0.483
precision : 0.000
recall    : 0.000
f1        : 0.000

=== BASELINE: BETO SIN FINE-TUNING (test, held-out) ===
accuracy  : 0.553
precision : 0.000
recall    : 0.000
f1        : 0.000


## 4. LoRA: fine-tuning eficiente


- `r` (rank): tamaño de las matrices pequeñas.
- `lora_alpha`: cuánto pesa el ajuste (regla común: `alpha ≈ 2×r`).
- `target_modules`: a qué capas se aplica. En BETO (BERT estándar) son `query` y `value`
  — en DistilBERT (el modelo del notebook base) son `q_lin` y `v_lin`.

In [10]:
from peft import LoraConfig, TaskType, get_peft_model

modelo_lora = cargar_modelo_base()

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    # BETO es un BERT estándar: sus capas de atención son "query"/"value", no
    # "q_lin"/"v_lin" (eso es específico de DistilBERT, el modelo del notebook base).
    target_modules=["query", "value"],
    modules_to_save=["classifier"],
)

modelo_lora = get_peft_model(modelo_lora, lora_cfg)
modelo_lora.to(device)

modelo_lora.print_trainable_parameters()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19700.02it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differe

trainable params: 296,450 || all params: 110,148,868 || trainable%: 0.2691


## 5. Entrenar con la Trainer API

La `Trainer` nos regala el loop de entrenamiento. Le damos el modelo, los datos, la métrica y unos
hiperparámetros mínimos. Entrena 2 épocas — con LoRA eso basta para ver el salto sobre el baseline.

In [11]:
from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metricas_clasificacion(logits, labels)

args_lora = TrainingArguments(
    output_dir="./lawten_lora_out",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,  # batch efectivo = 16
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_steps=19,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",

    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=modelo_lora,
    args=args_lora,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

resultado_entrenamiento = trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.413083,0.701458,0.494253,0.538462,0.155556,0.241379
2,1.334232,0.648471,0.620690,0.875000,0.311111,0.459016
3,1.157593,0.545887,0.781609,0.933333,0.622222,0.746667
4,1.027603,0.495918,0.793103,0.935484,0.644444,0.763158
5,0.707158,0.447409,0.793103,0.935484,0.644444,0.763158
6,0.827628,0.467836,0.793103,0.935484,0.644444,0.763158


## 6. Evaluación comparativa sobre validation (MODIFICADA)

La comparación principal de M1 se hace sobre el mismo `validation` usado durante el entrenamiento.
Se presentan juntos los dos baselines exigidos —clase mayoritaria y BETO sin afinar— y el modelo
BETO + LoRA. El resultado en `test` se conserva como información adicional del equipo, pero no
reemplaza la comparación solicitada sobre `validation`.

In [12]:
# Validation: mismo split usado por load_best_model_at_end / early stopping — sirve para
# seguir el entrenamiento, no es el número final "honesto".
metricas_lora_val = trainer.evaluate()

print("=== RESULTADOS EN VALIDATION ===")
for nombre in ["accuracy", "precision", "recall", "f1"]:
    base = metricas_baseline_val[nombre]
    afinado = metricas_lora_val[f"eval_{nombre}"]

    print(
        f"{nombre:10} | "
        f"baseline={base:.3f} | "
        f"LoRA={afinado:.3f} | "
        f"delta={afinado - base:+.3f}"
    )

# Test: held-out real, nunca visto durante entrenamiento ni selección de checkpoint.
# Este es el número que se reporta como resultado final de M1.
metricas_lora_test = trainer.evaluate(eval_dataset=test_tok)

print("\n=== RESULTADOS EN TEST (held-out) ===")
for nombre in ["accuracy", "precision", "recall", "f1"]:
    base = metricas_baseline_test[nombre]
    afinado = metricas_lora_test[f"eval_{nombre}"]

    print(
        f"{nombre:10} | "
        f"baseline={base:.3f} | "
        f"LoRA={afinado:.3f} | "
        f"delta={afinado - base:+.3f}"
    )

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.827628,0.495918,6,0.793103,0.935484,0.644444,0.763158


=== RESULTADOS EN VALIDATION ===
accuracy   | baseline=0.483 | LoRA=0.793 | delta=+0.310
precision  | baseline=0.000 | LoRA=0.935 | delta=+0.935
recall     | baseline=0.000 | LoRA=0.644 | delta=+0.644
f1         | baseline=0.000 | LoRA=0.763 | delta=+0.763


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.827628,0.355930,6,0.894737,0.906250,0.852941,0.878788



=== RESULTADOS EN TEST (held-out) ===
accuracy   | baseline=0.553 | LoRA=0.895 | delta=+0.342
precision  | baseline=0.000 | LoRA=0.906 | delta=+0.906
recall     | baseline=0.000 | LoRA=0.853 | delta=+0.853
f1         | baseline=0.000 | LoRA=0.879 | delta=+0.879


In [13]:
# TABLA COMPARATIVA DE BASELINES Y LORA (CREADA)
# Tabla principal de Samuel: los tres modelos evaluados sobre el mismo validation.
def fila_metricas(nombre_modelo, metricas):
    return {
        "modelo": nombre_modelo,
        "accuracy": metricas["accuracy"],
        "precision": metricas["precision"],
        "recall": metricas["recall"],
        "f1": metricas["f1"],
    }

metricas_lora_val_limpias = {
    nombre: metricas_lora_val[f"eval_{nombre}"]
    for nombre in ["accuracy", "precision", "recall", "f1"]
}

tabla_clasificacion_val = pd.DataFrame([
    fila_metricas("Clase mayoritaria", metricas_mayoritaria_val),
    fila_metricas("BETO sin afinar (zero-shot)", metricas_baseline_val),
    fila_metricas("BETO + LoRA", metricas_lora_val_limpias),
]).set_index("modelo")

print("=== COMPARACIÓN PRINCIPAL EN VALIDATION ===")
display(tabla_clasificacion_val.round(3))

delta_clasificacion_val = (
    tabla_clasificacion_val.loc["BETO + LoRA"]
    - tabla_clasificacion_val.loc["BETO sin afinar (zero-shot)"]
)
print("Mejora absoluta de LoRA frente a BETO sin afinar:")
display(delta_clasificacion_val.round(3).rename("delta").to_frame().T)

=== COMPARACIÓN PRINCIPAL EN VALIDATION ===


,accuracy,precision,recall,f1
modelo,,,,
Clase mayoritaria,0.483,0.000,0.000,0.000
BETO sin afinar (zero-shot),0.483,0.000,0.000,0.000
BETO + LoRA,0.793,0.935,0.644,0.763


Mejora absoluta de LoRA frente a BETO sin afinar:


,accuracy,precision,recall,f1
delta,0.31,0.935,0.644,0.763


## 7. Ranking por sentencia: Recall@k y Precision@k (MODIFICADA)

La clasificación binaria (sección 6) evalúa fila por fila. El producto, en cambio, reordena
candidatos *por sentencia*: importa si los artículos correctos quedan en el top-k después de
ordenar por score. La métrica principal acordada es `k=3` y se reporta sobre `validation`.
La clase mayoritaria figura como `N/A`: al dar el mismo score a todos los candidatos no define
un ranking, y cualquier orden dependería arbitrariamente del orden original de las filas.

In [14]:
def probabilidad_positiva(logits):
    logits = np.asarray(logits)
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    probabilidades = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probabilidades[:, 1]

def metricas_ranking(datos, logits, k=3):
    """
    Ordena los artículos candidatos de cada sentencia por score de relevancia.
    Luego calcula Recall@k y Precision@k, promediados por sentencia.
    """
    evaluacion = datos.copy().reset_index(drop=True)
    evaluacion["score_relevancia"] = probabilidad_positiva(logits)

    resultados = []

    for sentencia, grupo in evaluacion.groupby("sentencia_origen"):
        ranking = grupo.sort_values("score_relevancia", ascending=False)
        top_k = ranking.head(k)

        positivos_totales = grupo["label"].sum()
        positivos_en_top_k = top_k["label"].sum()

        # Si el join dejó una sentencia sin positivos en este split (su único positivo no
        # resolvió contra el diccionario), recall no está definido — se excluye del promedio.
        recall_at_k = (
            positivos_en_top_k / positivos_totales if positivos_totales > 0 else np.nan
        )

        resultados.append({
            "sentencia_origen": sentencia,
            "positivos_totales": positivos_totales,
            "positivos_en_top_k": positivos_en_top_k,
            "recall_at_k": recall_at_k,
            "precision_at_k": positivos_en_top_k / len(top_k),
        })

    resultados = pd.DataFrame(resultados)

    return {
        f"recall@{k}": resultados["recall_at_k"].mean(),
        f"precision@{k}": resultados["precision_at_k"].mean(),
        "detalle_por_sentencia": resultados,
    }

In [15]:
# EVALUACIÓN RECALL@K / PRECISION@K (MODIFICADA)
# Evaluación principal de ranking sobre el mismo validation usado por Miguel.
K_PRINCIPAL = 3

# BETO sin afinar ya produjo pred_baseline; aquí obtenemos los scores del LoRA final.
pred_lora_val = trainer.predict(val_tok)

ranking_baseline_val = metricas_ranking(
    val_df, pred_baseline.predictions, k=K_PRINCIPAL
)
ranking_lora_val = metricas_ranking(
    val_df, pred_lora_val.predictions, k=K_PRINCIPAL
)

col_recall = f"recall@{K_PRINCIPAL}"
col_precision = f"precision@{K_PRINCIPAL}"

tabla_ranking_val = pd.DataFrame([
    {"modelo": "Clase mayoritaria", col_recall: np.nan, col_precision: np.nan},
    {
        "modelo": "BETO sin afinar (zero-shot)",
        col_recall: ranking_baseline_val[col_recall],
        col_precision: ranking_baseline_val[col_precision],
    },
    {
        "modelo": "BETO + LoRA",
        col_recall: ranking_lora_val[col_recall],
        col_precision: ranking_lora_val[col_precision],
    },
]).set_index("modelo")

print(f"=== RANKING EN VALIDATION: k={K_PRINCIPAL} ===")
display(tabla_ranking_val.round(3))

delta_recall = ranking_lora_val[col_recall] - ranking_baseline_val[col_recall]
delta_precision = ranking_lora_val[col_precision] - ranking_baseline_val[col_precision]
print(f"Mejora absoluta LoRA vs. BETO sin afinar: {col_recall}={delta_recall:+.3f}, "
      f"{col_precision}={delta_precision:+.3f}")
print("Clase mayoritaria: N/A en ranking porque todos sus candidatos empatan en score.")

=== RANKING EN VALIDATION: k=3 ===


,recall@3,precision@3
modelo,,
Clase mayoritaria,NaN,NaN
BETO sin afinar (zero-shot),0.679,0.381
BETO + LoRA,0.898,0.587


Mejora absoluta LoRA vs. BETO sin afinar: recall@3=+0.219, precision@3=+0.206
Clase mayoritaria: N/A en ranking porque todos sus candidatos empatan en score.


In [16]:
detalle_val = ranking_lora_val["detalle_por_sentencia"]

display(
    detalle_val.sort_values("recall_at_k")
    .head(10)
)

,sentencia_origen,positivos_totales,positivos_en_top_k,recall_at_k,precision_at_k
3,SL-2858/22,5,3,0.600000,1.000000
4,SL-780/23,5,3,0.600000,1.000000
2,SL-2850/20,3,2,0.666667,0.666667
1,SL-1514/23,3,2,0.666667,0.666667
6,T-092/16,3,2,0.666667,0.666667
13,T-347/24,3,2,0.666667,0.666667
5,SU-428/23,3,3,1.000000,1.000000
0,SL-1050/23,1,1,1.000000,0.333333
7,T-1128/00,2,2,1.000000,0.666667
8,T-1136/00,1,1,1.000000,0.333333


## 8. Ejemplos cualitativos (qué hace el modelo) (MODIFICADA)

M1 pide al menos tres ejemplos de entrada → salida sobre `validation`. Para evitar escoger solo
los casos más favorables, la selección usa reglas explícitas: un verdadero positivo representativo,
un falso negativo cercano al umbral, un negativo difícil correctamente rechazado y, cuando exista,
el falso positivo con mayor score. El valor mostrado se llama **score de relevancia**, no confianza
calibrada. Todos los ejemplos conservan la consulta, el artículo y la etiqueta reales del dataset.

In [17]:
# FUNCIONES PARA MOSTRAR EJEMPLOS CUALITATIVOS (MODIFICADA)
def formatear_snippet(texto, max_chars=200):
    """Recorta un texto largo a un snippet legible sin cortar palabras a la mitad."""
    texto = " ".join(str(texto).split())
    if len(texto) <= max_chars:
        return texto
    return texto[:max_chars].rsplit(" ", 1)[0] + "..."

def mostrar_ejemplo(numero, fila):
    """
    Corre el modelo LoRA sobre una fila real de val_df/test_df — ya pasó por el join con
    el diccionario, así que el artículo citado y su texto son consistentes — y muestra la
    predicción junto al label real.
    """
    # Si la fila viene de val_scored reutilizamos el score con el que fue seleccionada.
    # El fallback permite seguir usando esta función con cualquier fila de val_df/test_df.
    if "score_relevancia" in fila.index:
        score = float(fila["score_relevancia"])
    else:
        entrada = tokenizer(
            fila["consulta"],
            fila["texto_input"],
            return_tensors="pt",
            truncation="only_second",
            max_length=512,
        ).to(device)

        trainer.model.eval()
        with torch.no_grad():
            logits = trainer.model(**entrada).logits
            score = torch.softmax(logits, dim=-1)[0, 1].item()

    prediccion = "RELEVANTE" if score >= 0.5 else "NO RELEVANTE"
    esperado = "RELEVANTE" if fila["label"] == 1 else "NO RELEVANTE"
    acierto = "✅ acierta" if prediccion == esperado else "❌ falla"
    caso = fila.get("caso_cualitativo", "Caso cualitativo")

    print(f"Ejemplo {numero} — {caso}")
    print(f"  Tipo     : {fila['tipo']}, sentencia: {fila['sentencia_origen']}")
    print(f"  Consulta : {formatear_snippet(fila['consulta'], 300)}")
    print(f"  Artículo : {fila['articulo']}")
    print(f"             {formatear_snippet(fila['texto_input'], 200)}")
    print(f"  Esperado : {esperado}")
    print(f"  Predicho : {prediccion} (score de relevancia={score:.3f})  {acierto}")
    print()

In [18]:
# SELECCIÓN DE EJEMPLOS CUALITATIVOS (MODIFICADA)
# Ejemplos reales de validation seleccionados con criterios reproducibles.
val_scored = val_df.copy()
val_scored["score_relevancia"] = probabilidad_positiva(pred_lora_val.predictions)
val_scored["prediccion"] = (val_scored["score_relevancia"] >= 0.5).astype(int)

verdaderos_positivos = val_scored[
    (val_scored["label"] == 1) & (val_scored["prediccion"] == 1)
].copy()
mediana_vp = verdaderos_positivos["score_relevancia"].median()
positivo_representativo = (
    verdaderos_positivos
    .assign(distancia_mediana=lambda datos: (datos["score_relevancia"] - mediana_vp).abs())
    .sort_values(["distancia_mediana", "sentencia_origen", "articulo"])
    .head(1)
    .drop(columns="distancia_mediana")
    .copy()
)
positivo_representativo["caso_cualitativo"] = "Verdadero positivo representativo"

falso_negativo = (
    val_scored[(val_scored["label"] == 1) & (val_scored["prediccion"] == 0)]
    .sort_values("score_relevancia", ascending=False)
    .head(1)
    .copy()
)
falso_negativo["caso_cualitativo"] = "Falso negativo más cercano al umbral"

negativo_dificil_correcto = (
    val_scored[
        (val_scored["tipo"] == "negativo_dificil_placeholder")
        & (val_scored["label"] == 0)
        & (val_scored["prediccion"] == 0)
    ]
    .sort_values("score_relevancia", ascending=False)
    .head(1)
    .copy()
)
negativo_dificil_correcto["caso_cualitativo"] = "Negativo difícil correctamente rechazado"

falso_positivo = (
    val_scored[(val_scored["label"] == 0) & (val_scored["prediccion"] == 1)]
    .sort_values("score_relevancia", ascending=False)
    .head(1)
    .copy()
)
if not falso_positivo.empty:
    falso_positivo["caso_cualitativo"] = "Falso positivo con mayor score"

selecciones = [
    caso for caso in [
        positivo_representativo,
        falso_negativo,
        negativo_dificil_correcto,
        falso_positivo,
    ]
    if not caso.empty
]

cantidad_seleccionada = sum(len(caso) for caso in selecciones)
if cantidad_seleccionada < 3:
    indices_usados = {indice for caso in selecciones for indice in caso.index}
    complementarios = (
        val_scored.loc[~val_scored.index.isin(indices_usados)]
        .assign(distancia_umbral=lambda datos: (datos["score_relevancia"] - 0.5).abs())
        .sort_values(["distancia_umbral", "sentencia_origen", "articulo"])
        .head(3 - cantidad_seleccionada)
        .drop(columns="distancia_umbral")
        .copy()
    )
    complementarios["caso_cualitativo"] = "Caso complementario cercano al umbral"
    selecciones.append(complementarios)

ejemplos_cualitativos = pd.concat(selecciones).reset_index(drop=True)

for numero, (_, fila) in enumerate(ejemplos_cualitativos.iterrows(), start=1):
    mostrar_ejemplo(numero, fila)

Ejemplo 1 — Verdadero positivo representativo
  Tipo     : positivo, sentencia: SL-2858/22
  Consulta : Trabajé para el ISS desde septiembre de 2000 hasta marzo de 2013 como contador público, pero me hicieron firmar contratos de prestación de servicios. Me despidieron sin justa causa y nunca me pagaron mis prestaciones ni derechos laborales. Trabajaba con horario fijo, recibía órdenes de un jefe y...
  Artículo : Ley 100 de 1993, Art. 275
             Ley 100 de 1993. ARTÍCULO 275. Del Instituto de Seguros Sociales. El Instituto de Seguros Sociales, es una empresa industrial y comercial del Estado, del orden nacional, con personería jurídica,...
  Esperado : RELEVANTE
  Predicho : RELEVANTE (score de relevancia=0.781)  ✅ acierta

Ejemplo 2 — Falso negativo más cercano al umbral
  Tipo     : positivo, sentencia: SU-428/23
  Consulta : Trabajé como odontóloga desde el año 2000. Sufrí dos accidentes laborales que me causaron una enfermedad profesional en la mano derecha (Tenosinovitis). M

## 9. Guardar el adaptador LoRA (opcional)

LoRA solo guarda las matrices pequeñas: son unos pocos MB, no el modelo entero.

In [19]:
modelo_lora.save_pretrained('./s04_lora_adapter')
print('Adaptador LoRA guardado en ./s04_lora_adapter (solo los pesos de LoRA).')

Adaptador LoRA guardado en ./s04_lora_adapter (solo los pesos de LoRA).
